# Unlocking Sports Betting with Python in 2026
**Updated guide to The Odds API v4** — pulling live odds, comparing bookmakers, and finding value with Python.

This is a 2026 refresh of the original [Unlocking Sports Betting with Python](https://medium.com/@ben.g.ballard/unlocking-sports-betting-with-python-the-odds-api-3bc74eb24153) (Dec 2023).

In [ ]:
# Setup
import os
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Resolve .env relative to this notebook's location
notebook_dir = Path(__file__).parent if '__file__' in dir() else Path.cwd()
env_path = Path(__file__).parent.parent / '.env' if '__file__' in dir() else None

# Try multiple possible locations for the .env file
for candidate in [
    Path(notebook_dir, '..', '.env'),       # sibling of notebook's parent
    Path('Medium', '.env'),                  # from workspace root
    Path('..', '.env'),                      # one level up from cwd
]:
    if candidate.resolve().exists():
        load_dotenv(candidate.resolve())
        break

API_KEY = os.getenv('ODDS_API_KEY')
BASE_URL = 'https://api.the-odds-api.com/v4'

print(f"API key loaded: {'Yes' if API_KEY else 'No — check .env path!'}")

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## Step 1: Explore Available Sports

The `/sports` endpoint returns every sport currently in-season. This is your starting point — it tells you which `sport_key` values to use in subsequent calls.

In [ ]:
# Fetch all in-season sports
response = requests.get(
    f'{BASE_URL}/sports',
    params={'apiKey': API_KEY}
)
response.raise_for_status()

sports = response.json()

# Show quota usage from response headers
print(f"Requests used: {response.headers.get('x-requests-used')}")
print(f"Requests remaining: {response.headers.get('x-requests-remaining')}")
print(f"\nIn-season sports: {len([s for s in sports if s['active']])}\n")

sports_df = pd.DataFrame(sports)
sports_df = sports_df[sports_df['active'] == True][['key', 'group', 'title', 'description']]
sports_df.head(15)

## Step 2: Pull Live Odds for a Sport

Now we'll grab live moneyline (h2h), spread, and totals odds for NBA games. The key parameters:
- **`regions`** — `us` for American bookmakers (DraftKings, FanDuel, etc.)
- **`markets`** — `h2h` (moneyline), `spreads`, `totals`
- **`oddsFormat`** — `american` for +/- lines familiar to US bettors

In [ ]:
SPORT = 'basketball_nba'

response = requests.get(
    f'{BASE_URL}/sports/{SPORT}/odds',
    params={
        'apiKey': API_KEY,
        'regions': 'us',
        'markets': 'h2h,spreads,totals',
        'oddsFormat': 'american',
        'dateFormat': 'iso',
    }
)
response.raise_for_status()

events = response.json()
print(f"Events returned: {len(events)}")
print(f"Requests remaining: {response.headers.get('x-requests-remaining')}")

# Peek at the first event structure
if events:
    event = events[0]
    print(f"\nExample: {event['home_team']} vs {event['away_team']}")
    print(f"Start: {event['commence_time']}")
    print(f"Bookmakers: {len(event['bookmakers'])}")

## Step 3: Wrangle the Nested JSON into a Clean DataFrame

The API response is deeply nested — each event has bookmakers, and each bookmaker has markets with outcomes. We'll flatten this into a tidy DataFrame.

In [ ]:
def flatten_odds(events):
    """Flatten the nested API response into a tidy DataFrame."""
    rows = []
    for event in events:
        game = f"{event['away_team']} @ {event['home_team']}"
        for bookmaker in event['bookmakers']:
            for market in bookmaker['markets']:
                for outcome in market['outcomes']:
                    row = {
                        'game': game,
                        'home_team': event['home_team'],
                        'away_team': event['away_team'],
                        'commence_time': event['commence_time'],
                        'bookmaker': bookmaker['title'],
                        'market': market['key'],
                        'outcome': outcome['name'],
                        'price': outcome['price'],
                    }
                    # Spreads and totals have a 'point' field
                    if 'point' in outcome:
                        row['point'] = outcome['point']
                    rows.append(row)
    return pd.DataFrame(rows)

odds_df = flatten_odds(events)
odds_df['commence_time'] = pd.to_datetime(odds_df['commence_time'])
print(f"Shape: {odds_df.shape}")
odds_df.head(10)

## Step 4: Compare Moneyline Odds Across Bookmakers

One of the most valuable things you can do with odds data is compare lines across bookmakers. Different books set different prices — and the gap is where value lives.

In [ ]:
# Filter to moneyline (h2h) odds only
ml_df = odds_df[odds_df['market'] == 'h2h'].copy()

# Pivot: one row per game/team, one column per bookmaker
ml_pivot = ml_df.pivot_table(
    index=['game', 'outcome'],
    columns='bookmaker',
    values='price',
    aggfunc='first'
)

# Add a column for the best (highest) line available
ml_pivot['best_odds'] = ml_pivot.max(axis=1)
ml_pivot['worst_odds'] = ml_pivot.min(axis=1)
ml_pivot['spread_gap'] = ml_pivot['best_odds'] - ml_pivot['worst_odds']

ml_pivot.sort_values('spread_gap', ascending=False).head(10)

## Step 5: Visualize — Moneyline Comparison by Bookmaker

Let's pick the first upcoming game and chart how each bookmaker prices the moneyline.

In [ ]:
# Pick the first game for the visualization
first_game = ml_df['game'].iloc[0]
game_ml = ml_df[ml_df['game'] == first_game]

fig, ax = plt.subplots(figsize=(12, 6))

teams = game_ml['outcome'].unique()
colors = ['#1f77b4', '#ff7f0e']
x = np.arange(len(game_ml['bookmaker'].unique()))
width = 0.35

for i, team in enumerate(teams):
    team_data = game_ml[game_ml['outcome'] == team].sort_values('bookmaker')
    bars = ax.bar(x + i * width, team_data['price'], width,
                  label=team, color=colors[i], edgecolor='white')
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        sign = '+' if height > 0 else ''
        ax.text(bar.get_x() + bar.get_width() / 2., height,
                f'{sign}{int(height)}', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_xlabel('Bookmaker')
ax.set_ylabel('American Odds')
ax.set_title(f'Moneyline Odds Comparison: {first_game}', fontsize=14, fontweight='bold')
ax.set_xticks(x + width / 2)
bookmakers = game_ml[game_ml['outcome'] == teams[0]].sort_values('bookmaker')['bookmaker'].values
ax.set_xticklabels(bookmakers, rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('images/moneyline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6: Convert Odds to Implied Probability

American odds look intimidating, but they translate directly to implied probability — the bookmaker's estimated chance of each outcome. This is the foundation of finding value bets.

In [ ]:
def american_to_implied_prob(odds):
    """Convert American odds to implied probability."""
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return abs(odds) / (abs(odds) + 100)

# Add implied probability to our moneyline data
ml_df['implied_prob'] = ml_df['price'].apply(american_to_implied_prob)

# For each game, calculate the average implied probability across bookmakers
avg_probs = ml_df.groupby(['game', 'outcome'])['implied_prob'].agg(['mean', 'std']).reset_index()
avg_probs.columns = ['game', 'team', 'avg_prob', 'prob_std']
avg_probs['avg_prob_pct'] = (avg_probs['avg_prob'] * 100).round(1)

avg_probs.sort_values(['game', 'avg_prob'], ascending=[True, False])

## Step 7: Visualize — Implied Win Probability by Game

A horizontal bar chart showing each team's implied probability makes it easy to see at a glance which games are close and which are lopsided.

In [ ]:
# Build a diverging bar chart of implied probabilities
games = avg_probs['game'].unique()
fig, ax = plt.subplots(figsize=(12, max(4, len(games) * 0.8)))

for i, game in enumerate(games):
    game_data = avg_probs[avg_probs['game'] == game].sort_values('avg_prob', ascending=False)
    if len(game_data) < 2:
        continue
    fav = game_data.iloc[0]
    dog = game_data.iloc[1]

    ax.barh(i, fav['avg_prob_pct'], color='#2ecc71', height=0.6, label='Favorite' if i == 0 else '')
    ax.barh(i, -dog['avg_prob_pct'], color='#e74c3c', height=0.6, label='Underdog' if i == 0 else '')

    ax.text(fav['avg_prob_pct'] + 1, i, f"{fav['team']} ({fav['avg_prob_pct']}%)",
            va='center', fontsize=10, fontweight='bold')
    ax.text(-dog['avg_prob_pct'] - 1, i, f"{dog['team']} ({dog['avg_prob_pct']}%)",
            va='center', ha='right', fontsize=10, fontweight='bold')

ax.set_yticks(range(len(games)))
ax.set_yticklabels([g.replace(' @ ', '\n@ ') for g in games], fontsize=9)
ax.set_xlabel('Implied Win Probability (%)')
ax.set_title('NBA Implied Win Probabilities (Avg Across Bookmakers)', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('images/implied_probability.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: NEW in 2026 — Player Props

Since the original article, The Odds API has expanded significantly. One of the biggest additions is **player prop markets** — points, rebounds, assists, threes, and more. These use the event-level endpoint.

In [ ]:
# First, get the list of events to grab an event ID
events_response = requests.get(
    f'{BASE_URL}/sports/{SPORT}/events',
    params={
        'apiKey': API_KEY,
        'dateFormat': 'iso',
    }
)
events_response.raise_for_status()
event_list = events_response.json()

if event_list:
    event_id = event_list[0]['id']
    event_name = f"{event_list[0]['away_team']} @ {event_list[0]['home_team']}"
    print(f"Fetching player props for: {event_name}")
    print(f"Event ID: {event_id}")

    # Fetch player points props for this event
    props_response = requests.get(
        f'{BASE_URL}/sports/{SPORT}/events/{event_id}/odds',
        params={
            'apiKey': API_KEY,
            'regions': 'us',
            'markets': 'player_points,player_rebounds,player_assists',
            'oddsFormat': 'american',
            'dateFormat': 'iso',
        }
    )
    props_response.raise_for_status()
    props_data = props_response.json()
    print(f"Requests remaining: {props_response.headers.get('x-requests-remaining')}")

In [ ]:
def flatten_props(event_data):
    """Flatten player props response into a DataFrame."""
    rows = []
    for bookmaker in event_data.get('bookmakers', []):
        for market in bookmaker['markets']:
            for outcome in market['outcomes']:
                rows.append({
                    'bookmaker': bookmaker['title'],
                    'market': market['key'],
                    'player': outcome.get('description', outcome['name']),
                    'side': outcome['name'],  # Over or Under
                    'line': outcome.get('point', None),
                    'price': outcome['price'],
                })
    return pd.DataFrame(rows)

props_df = flatten_props(props_data)
print(f"Player prop lines: {props_df.shape[0]}")
print(f"Markets: {props_df['market'].unique()}")
print(f"Players: {props_df['player'].nunique()}")
props_df.head(10)

## Step 9: Visualize — Player Points Lines Across Books

Where bookmakers disagree on a player's point total, there's potential value. Let's compare point lines for the top players.

In [ ]:
# Focus on player_points, Over side only (to compare lines)
points_over = props_df[
    (props_df['market'] == 'player_points') & (props_df['side'] == 'Over')
].copy()

# Get players with lines from multiple bookmakers
player_counts = points_over.groupby('player')['bookmaker'].nunique()
multi_book_players = player_counts[player_counts >= 3].index

plot_data = points_over[points_over['player'].isin(multi_book_players)]

# Sort by average line (highest scoring players first)
player_order = plot_data.groupby('player')['line'].mean().sort_values(ascending=True).index

fig, ax = plt.subplots(figsize=(12, max(4, len(player_order) * 0.5)))

for i, player in enumerate(player_order):
    player_data = plot_data[plot_data['player'] == player]
    lines = player_data['line'].values
    ax.scatter(lines, [i] * len(lines), s=80, zorder=3, alpha=0.7)
    ax.plot([lines.min(), lines.max()], [i, i], color='gray', linewidth=2, alpha=0.5)
    if lines.max() - lines.min() > 0:
        ax.annotate(f"Δ {lines.max() - lines.min():.1f}",
                    xy=(lines.max() + 0.3, i), fontsize=9, color='red', fontweight='bold')

ax.set_yticks(range(len(player_order)))
ax.set_yticklabels(player_order, fontsize=10)
ax.set_xlabel('Points Line (Over)')
ax.set_title(f'Player Points Props: Line Variance Across Bookmakers\n{event_name}',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('images/player_props_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusions

**What we built:**
1. Pulled live sports from The Odds API v4
2. Fetched moneyline, spread, and totals odds for NBA games
3. Flattened nested JSON into clean pandas DataFrames
4. Compared bookmaker pricing to find gaps
5. Converted American odds to implied probability
6. **NEW:** Pulled player prop markets (points, rebounds, assists) and compared lines across books

**What's next:** In the next article, we'll build a Streamlit dashboard that auto-refreshes these odds and highlights the biggest bookmaker discrepancies in real time.